In [ ]:
import h5py
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
from datetime import date


In [ ]:
# ------------------------------------------------------------------
# 1. Files to scan.
#
# Trajectum pT-study output, one file per energy, 200 MeV track cut.
# Only the 0-5% centrality bin is used in this notebook.
#
# 11.5 GeV is newly added to match the STAR HEPData point at the same
# energy -- update the path below once that run/export exists.
# ------------------------------------------------------------------
filepaths = [
    "src/monotonic_ptfluc_study/7.7GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/11.5GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/19.6GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/27GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/54GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/200GeV_200MeVcut_ptfluc_study.h5",
]

# Same physics setup, but with NO SMASH afterburner / rehadronizer
# step. Paired 1:1 by energy with `filepaths` above (same ordering,
# same 200 MeV track cut).
filepaths_nosmash = [
    "src/monotonic_ptfluc_nosmash_study/7.7GeV_200MeVcut_nosmash_ptfluc_study.h5",
    "src/monotonic_ptfluc_nosmash_study/11.5GeV_200MeVcut_nosmash_ptfluc_study.h5",
    "src/monotonic_ptfluc_nosmash_study/19.6GeV_200MeVcut_nosmash_ptfluc_study.h5",
    "src/monotonic_ptfluc_nosmash_study/27GeV_200MeVcut_nosmash_ptfluc_study.h5",
    "src/monotonic_ptfluc_nosmash_study/54GeV_200MeVcut_nosmash_ptfluc_study.h5",
    "src/monotonic_ptfluc_nosmash_study/200GeV_200MeVcut_nosmash_ptfluc_study.h5",
]

# Trajectum nominal energy -> matching STAR sqrt(s_NN) label in the
# HEPData table (Table 53 from Fig 5, ins1712047). Only these energies
# have a STAR 0-5% point to compare against.
STAR_ENERGY_MATCH = {
    7.7: 7.7,
    11.5: 11.5,
    19.0: 19.6,
    27.0: 27.0,
    200.0: 200.0,
}

star_csv_path = "HEPData-ins1712047-v1-Table_53_from_Fig_5.csv"


In [ ]:
# ------------------------------------------------------------------
# 2. Helper: process one file & return POINTS at the 0-5% centrality
#    bin only (all estimators).
#
# Trajectum's ptfluctuationscharged output is already normalized by
# mean pT, so no separate division by meanptcharged is needed here.
# ------------------------------------------------------------------
CENT_IDX_05 = 0  # 0-5%


def _as_2d(a):
    a = np.asarray(a)
    return a[:, None] if a.ndim == 1 else a


def _energy_from_filename(path):
    m = re.search(r"(\d+(?:\.\d+)?)GeV", str(path))
    if not m:
        raise ValueError(f"Couldn't parse energy from filename: {path}")
    return float(m.group(1))


def _detect_track_group(hdf, base="meanptcharged"):
    """Return the single track-cut group name under `base`
    (e.g. "STARTPC", "STARTPC150", "STARTPC200MeV"). Raises if the
    file contains more than one, in which case pass startpc_group
    explicitly to get_0_5_points_from_file."""
    groups = list(hdf[base].keys())
    if len(groups) != 1:
        raise ValueError(
            f"Expected exactly one track-cut group under '{base}', "
            f"found {groups}. Pass startpc_group explicitly."
        )
    return groups[0]


def get_0_5_points_from_file(filepath, startpc_group=None):
    """
    Returns records for the 0-5% centrality bin only:
      (energy_GeV, estimator_idx, y, yerr, track_group)
    """
    E = _energy_from_filename(filepath)
    points = []

    with h5py.File(filepath, "r") as hdf:
        grp = startpc_group or _detect_track_group(hdf)

        fluc_base = f"ptfluctuationscharged/{grp}/centralitybinned"

        dp_vals = _as_2d(hdf[f"{fluc_base}/values"][:])
        dp_uerr = _as_2d(hdf[f"{fluc_base}/uppererrors"][:])
        dp_lerr = _as_2d(hdf[f"{fluc_base}/lowererrors"][:])
        dp_sym = 0.5 * (dp_uerr + dp_lerr)

        with np.errstate(divide="ignore", invalid="ignore"):
            yvals = dp_vals * 100.0
            yerr = yvals * np.sqrt((dp_sym / dp_vals) ** 2)

        n_est = yvals.shape[1]

        for est in range(n_est):
            y = float(yvals[CENT_IDX_05, est])
            ye = float(yerr[CENT_IDX_05, est])
            if np.isfinite(y) and np.isfinite(ye):
                points.append((E, est, y, ye, grp))

    return points


In [ ]:
# ------------------------------------------------------------------
# 3. Load Trajectum points (0-5% only) and the STAR HEPData table.
#
# ESTIMATOR_IDX picks which pT-fluctuation estimator to compare --
# defaulting to 0. Change this if a different estimator is the
# STAR-equivalent one.
# ------------------------------------------------------------------
ESTIMATOR_IDX = 0

all_points = []
for fp in filepaths:
    all_points.extend(get_0_5_points_from_file(fp))

traj_df = pd.DataFrame(
    all_points,
    columns=["energy_GeV", "estimator", "value_percent", "error_percent", "track_group"],
)
traj_df = traj_df.sort_values(["track_group", "estimator", "energy_GeV"]).reset_index(drop=True)

all_points_nosmash = []
for fp in filepaths_nosmash:
    all_points_nosmash.extend(get_0_5_points_from_file(fp))

traj_nosmash_df = pd.DataFrame(
    all_points_nosmash,
    columns=["energy_GeV", "estimator", "value_percent", "error_percent", "track_group"],
)
traj_nosmash_df = traj_nosmash_df.sort_values(["track_group", "estimator", "energy_GeV"]).reset_index(drop=True)

star_df = pd.read_csv(star_csv_path, comment="#")
star_df.columns = ["sqrt_s_NN", "value_percent", "stat_plus", "stat_minus", "sys_plus", "sys_minus"]
star_df["error_percent"] = np.sqrt(star_df["stat_plus"] ** 2 + star_df["sys_plus"] ** 2)

traj_df.head()


In [ ]:
# ------------------------------------------------------------------
# 4. Single combined plot: Trajectum (full), Trajectum (no SMASH), and
#    STAR, 0-5% centrality, points only (no connecting lines), plotted
#    against sqrt(s_NN) on a LOG x-axis.
#
# No energy matching/pairing here -- each dataset is plotted at its
# own actual sqrt(s_NN), independently. STAR shows ALL its points
# (not just ones with a corresponding Trajectum energy).
#
# Styling modeled after the STAR BES-I dp_T/<p_T> summary plot
# (dotted gridlines, inward ticks on all 4 sides, minor ticks,
# black-edged filled markers, legend box).
# ------------------------------------------------------------------
from matplotlib.ticker import AutoMinorLocator

plt.rcParams.update({
    "text.usetex": False,
    "mathtext.fontset": "cm",
    "font.family": "serif",
    "font.size": 14,
})

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

traj_plot = traj_df[traj_df["estimator"] == ESTIMATOR_IDX].sort_values("energy_GeV")
ax.errorbar(traj_plot["energy_GeV"], traj_plot["value_percent"], yerr=traj_plot["error_percent"],
            fmt="s", color="tab:blue", markersize=9, markeredgecolor="black", markeredgewidth=0.8,
            ecolor="tab:blue", elinewidth=1.2, capsize=4, capthick=1.2, label="Trajectum")

nosmash_plot = traj_nosmash_df[traj_nosmash_df["estimator"] == ESTIMATOR_IDX].sort_values("energy_GeV")
ax.errorbar(nosmash_plot["energy_GeV"], nosmash_plot["value_percent"], yerr=nosmash_plot["error_percent"],
            fmt="^", color="tab:orange", markersize=9, markeredgecolor="black", markeredgewidth=0.8,
            ecolor="tab:orange", elinewidth=1.2, capsize=4, capthick=1.2, label="Trajectum (no SMASH)")

star_plot = star_df.sort_values("sqrt_s_NN")
ax.errorbar(star_plot["sqrt_s_NN"], star_plot["value_percent"], yerr=star_plot["error_percent"],
            fmt="o", color="tab:red", markersize=9, markeredgecolor="black", markeredgewidth=0.8,
            ecolor="tab:red", elinewidth=1.2, capsize=4, capthick=1.2, label="STAR")

ax.set_xscale("log")
ax.set_xlabel(r"$\sqrt{s_{NN}}$  [GeV]", fontsize=15)
ax.set_ylabel(r"$\sqrt{\langle \Delta p_{T,i}\Delta p_{T,j}\rangle}/\langle p_T\rangle$  [%]", fontsize=15)

# Dotted gridlines, major + minor, matching the reference style
ax.grid(True, which="major", linestyle=":", linewidth=0.8, color="0.5", alpha=0.7)
ax.grid(True, which="minor", linestyle=":", linewidth=0.5, color="0.7", alpha=0.4)
ax.yaxis.set_minor_locator(AutoMinorLocator())

# Inward ticks on all four sides, major + minor
ax.tick_params(axis="both", which="major", direction="in", length=7, width=1.0,
                top=True, right=True, labelsize=13)
ax.tick_params(axis="both", which="minor", direction="in", length=4, width=0.8,
                top=True, right=True)

for spine in ax.spines.values():
    spine.set_linewidth(1.2)

legend = ax.legend(loc="lower right", fontsize=12, frameon=True, framealpha=0.9,
                    edgecolor="0.6", handletextpad=0.6)

fig.suptitle(r"Trajectum vs STAR: $\delta p_T/\langle p_T\rangle$, 0-5% centrality", fontsize=15)
fig.tight_layout()
fig.savefig("graphs/pt_fluctuations_vs_STAR_0-5pct.png")
plt.show()
